# Newton 101: From Concepts to Manipulation

This notebook is a guided, hands-on introduction to Newton for readers with some robotics/simulation/RL background who may be new to Warp. We start with high-level ideas, then build practical intuition with small interactive simulations, and finish with a simple Franka pick-and-place task using inverse kinematics.

<span style="background:#ffc">🎬 See also the Newton [hands-on lab from GTC 2026](https://www.nvidia.com/en-us/on-demand/session/gtc26-dlit81700/) that covered a similar version of this notebook.</span>

## Roadmap

1. **Core concepts**: ModelBuilder → Model → State / Control → Solver
2. **A tiny simulation** with visualization and a simple loop
3. **Performance**: CUDA graph capture (when running on GPU)
4. **Load a robot + joint forces** with `Control.joint_f`
5. **Joint targets** with `Control.joint_target_pos`
6. **IK path following**: draw a rectangle in front of the robot
7. **Manipulation**: Franka pick-and-place with IK + gripper control

## Setup and Imports

We use Warp for numerical compute and Newton for simulation. We also use the `ViewerViser` backend so each notebook visualization cell can save a replayable recording with timeline controls.

In [ ]:
import numpy as np
import warp as wp

import newton
import newton.examples
import newton.utils
import newton.ik as ik

wp.config.quiet = True

### Utilities

We'll keep names simple and re-use them later. The `make_viewer()` helper sets up a Viser recording that is playable inside the notebook.

In [ ]:
# Helper functions

from html import escape
import json
import os
from pathlib import Path
import time
import uuid

from IPython.display import HTML, Javascript, display

# Optional: force CPU for debugging (default is GPU if available)
use_cpu = False
if use_cpu:
    wp.set_device("cpu")


def make_viewer(name: str):
    """Create a replayable Viser recording for an output cell."""
    recording_path = Path("../_static/recordings") / f"{name}.viser"
    recording_path.parent.mkdir(parents=True, exist_ok=True)
    viewer = newton.viewer.ViewerViser(verbose=False, record_to_viser=str(recording_path))
    return viewer


def render_mermaid(diagram: str, theme: str = "forest", line_color: str = "#76b900"):
    """Render Mermaid text as a diagram in Jupyter output."""
    element_id = f"mermaid-{uuid.uuid4().hex}"
    display(HTML(f'<div id="{element_id}" class="mermaid">{escape(diagram)}</div>'))

    config_json = json.dumps(
        {
            "startOnLoad": False,
            "theme": theme,
            "themeVariables": {"lineColor": line_color},
        }
    )

    js = f"""
(async () => {{
  const loadMermaid = () => new Promise((resolve, reject) => {{
    if (window.mermaid) {{
      resolve();
      return;
    }}
    const script = document.createElement("script");
    script.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    script.onload = () => resolve();
    script.onerror = () => reject(new Error("Failed to load Mermaid."));
    document.head.appendChild(script);
  }});

  await loadMermaid();
  const config = {config_json};
  window.mermaid.initialize(config);

  const node = document.getElementById("{element_id}");
  if (!node) {{
    return;
  }}

  try {{
    await window.mermaid.run({{ nodes: [node] }});
  }} catch (err) {{
    node.textContent = "Mermaid render error: " + (err && err.message ? err.message : err);
  }}
}})();
"""
    display(Javascript(js))



class _HTMLProgressBar:
    """Frontend-agnostic HTML progress bar for notebook loops."""

    def __init__(self, iterable, total: int, desc: str = "", leave: bool = True):
        self._iterable = iterable
        self.total = max(int(total), 1)
        self.desc = desc
        self.leave = leave
        self.n = 0
        self._start = time.perf_counter()
        self._last_refresh = 0.0
        self._handle = display(HTML(self._render()), display_id=True)

    def _render(self) -> str:
        elapsed = max(time.perf_counter() - self._start, 1e-9)
        pct = min(100.0, 100.0 * self.n / self.total)
        rate = self.n / elapsed
        remaining = (self.total - self.n) / rate if rate > 1e-9 else float("inf")
        eta = "--:--" if not np.isfinite(remaining) else f"{int(remaining // 60):02d}:{int(remaining % 60):02d}"
        desc_html = escape(self.desc)
        return (
            f'<div style="font-family: sans-serif; margin: 6px 0;">'
            f'<div style="display:flex; justify-content:space-between; font-size:12px; margin-bottom:4px;">'
            f'<span>{desc_html}</span>'
            f'<span>{self.n}/{self.total} ({pct:5.1f}%)</span>'
            f'</div>'
            f'<progress value="{self.n}" max="{self.total}" style="width:100%; height:14px;"></progress>'
            f'<div style="font-size:11px; color:#666; margin-top:2px;">elapsed {elapsed:5.1f}s | eta {eta}</div>'
            f'</div>'
        )

    def _refresh(self, force: bool = False):
        now = time.perf_counter()
        if force or (now - self._last_refresh) >= 0.1 or self.n >= self.total:
            self._handle.update(HTML(self._render()))
            self._last_refresh = now

    def set_description(self, desc: str):
        self.desc = desc
        self._refresh(force=True)

    def set_description_str(self, desc: str):
        self.set_description(desc)

    def update(self, n: int = 1):
        self.n = min(self.total, self.n + int(n))
        self._refresh()

    def close(self):
        if self.leave:
            self._refresh(force=True)
        else:
            self._handle.update(HTML(""))

    def __iter__(self):
        try:
            for item in self._iterable:
                yield item
                self.update(1)
        finally:
            self.close()


def _tqdm_html(iterable=None, total=None, desc: str = "", leave: bool = False, **_kwargs):
    if iterable is None:
        if total is None:
            raise ValueError("Either iterable or total must be provided.")
        iterable = range(int(total))
    if total is None:
        try:
            total = len(iterable)
        except TypeError as e:
            raise ValueError("total is required for iterables without len().") from e
    return _HTMLProgressBar(iterable=iterable, total=int(total), desc=desc, leave=leave)


def _trange_html(*args, **kwargs):
    return _tqdm_html(range(*args), **kwargs)


# Use HTML progress bars for reliable updates in this frontend.
tqdm, trange = _tqdm_html, _trange_html

## Core Concepts

Newton simulations revolve around a small set of core abstractions:

- **ModelBuilder**: construction API for importing assets from USD, MJCF, URDF, assembling bodies, shapes, joints, and their properties.
- **Model**: the compiled, simulation-ready data (arrays and metadata) that lives on the compute device.
- **State**: time-varying data (positions, velocities, forces).
- **Control**: control inputs for joints (target positions, torques, etc.).
- **Contacts**: geometric contact information generated by the collision detection pipeline.
- **Solver**: advances the simulation by integrating physics and handling constraints.

In each simulation substep, the solver consumes model/state/control/contacts plus `dt`, and writes the next state.

## Newton is based on NVIDIA Warp

Warp is NVIDIA’s Python framework for writing high-performance simulation and geometry code without dropping into CUDA by hand. You write kernels in Python (for example with `@wp.kernel`), and Warp JIT-compiles them into optimized native code that runs on CPU or GPU. In practice, it gives you a Python workflow with near low-level performance for physics, robotics, and graphics workloads.

In [ ]:
warp_workflow_diagram = r"""
flowchart LR
    K["Write @wp.kernel in Python"]
    L["First wp.launch(..., device='cuda')"]

    subgraph JIT["Warp JIT"]
        AST["Parse kernel + types"]
        CU["Generate CUDA C++"]
        NV["Compile with NVRTC/NVCC"]
    end

    RUN["Run kernel on GPU"]

    K --> L --> AST --> CU --> NV --> RUN
"""

render_mermaid(warp_workflow_diagram, theme="forest", line_color="#76b900")

## 1) ModelBuilder → Model

Let's build a minimal scene: a ground plane, one box, and one sphere. This keeps things simple while we explore the API.

In this section we also allocate `State`, `Control`, and a `Contacts` object, which is then passed into solver stepping.


In [ ]:
core_concepts_diagram = r"""
flowchart LR
    subgraph Input["Input Data"]
        M[newton.Model]
        S[newton.State]
        C[newton.Control]
        K[newton.Contacts]
        DT[Time step dt]
    end

    STEP["solver.step()"]

    subgraph Output["Output Data"]
        SO["newton.State (updated)"]
    end

    %% Connections
    M --> STEP
    S --> STEP
    C --> STEP
    K --> STEP
    DT --> STEP
    STEP --> SO
"""

render_mermaid(core_concepts_diagram, theme="forest", line_color="#76b900")

In [ ]:
builder = newton.ModelBuilder()

# Static ground (body=-1 means it's attached to the world, i.e. static)
builder.add_ground_plane()

# Dynamic box - add_body() returns the body index, which we pass to add_shape_*()
box_body = builder.add_body(xform=wp.transform(wp.vec3(0.0, -1.0, 1.0), wp.quat_identity()))
builder.add_shape_box(box_body, hx=0.15, hy=0.15, hz=0.15)

# Dynamic sphere
sphere_body = builder.add_body(xform=wp.transform(wp.vec3(0.0, 1.0, 1.2), wp.quat_identity()))
builder.add_shape_sphere(sphere_body, radius=0.18)

print(f"Bodies: {builder.body_count}, Shapes: {builder.shape_count}")

### Finalize → Model, State, Control, Contacts

Finalizing converts the builder into a device-ready `Model`. From that model we allocate two `State` objects, one `Control`, and one reusable `Contacts` buffer.

`Contacts` is populated by the collision pipeline at each substep and passed into `solver.step(...)` alongside state, control, and `dt`.

In [ ]:
# finalize() compiles the builder into a device-ready Model
model = builder.finalize()

# Allocate two State buffers (we ping-pong between them during simulation)
state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

print(f"Model device: {model.device}")
print(f"Body count: {model.body_count}")

viewer = make_viewer("tiny_model")
viewer.set_model(model)
viewer.log_state(state_0)
viewer

_Note: you can orbit the camera in the viewer by dragging the mouse while pressing the left mouse button.
Hold the right mouse button while dragging to pan the camera, use the mouse wheel to zoom._

## 2) Solver and Simulation Loop

We will use the XPBD solver for this tiny scene. The loop is always the same pattern: clear forces, collide, step, swap state buffers.

In [ ]:
solver = newton.solvers.SolverXPBD(model, iterations=10)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps


def simulate():
    """Run one frame of simulation (multiple substeps)."""
    global state_0, state_1

    for _ in range(sim_substeps):
        state_0.clear_forces()
        contacts = model.collide(state_0)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        # Swap buffers so state_0 always holds the latest state
        state_0, state_1 = state_1, state_0

### Visualize the First Simulation

We will record a short clip using `ViewerViser` so it can be replayed in the notebook.

In [ ]:
viewer = make_viewer("01_simple_scene")
viewer.set_model(model)

print("Starting simulation, this may take a minute or longer to compile the CUDA kernels...")

sim_time = 0.0
num_frames = 180
for _ in trange(num_frames):
    simulate()
    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.end_frame()
    sim_time += frame_dt

viewer

## 3) Performance: CUDA Graph Capture

On GPU, repeatedly launching many tiny kernels can dominate runtime. Warp supports CUDA graph capture to reduce overhead. We will wrap our `simulate()` loop once and re-use it.

<img src="images/cuda-graph.png" width="700">

In [ ]:
# NOTE: CUDA graphs capture the exact GPU operations in simulate() as a replayable graph.
# Do not reallocate state_0/state_1 after capture - the graph references those specific buffers.
graph = None
if wp.get_device().is_cuda:
    with wp.ScopedCapture() as capture:
        simulate()
    graph = capture.graph
    print("CUDA graph captured - kernel launches are now batched into a single replay")
else:
    print("Running on CPU; graph capture skipped")

Let's compare the performance with and without graph capture:

In [ ]:
if wp.get_device().is_cuda:
    with wp.ScopedTimer("without_graph_capture"):
        for _ in range(120):
            simulate()

In [ ]:
if graph is not None:
    with wp.ScopedTimer("with_graph_capture"):
        for _ in range(120):
            wp.capture_launch(graph)

## 4) Load a Robot + Apply Joint Forces

We will load a Franka arm from URDF and apply a sinusoidal torque using `Control.joint_f`. This is the lowest-level control interface, useful for direct torque control or learned policies.

In [ ]:
# Franka arm initial joint positions (7 arm joints + 2 gripper fingers)
FRANKA_HOME_Q = [
    -3.680e-03, 2.390e-02, 3.680e-03, -2.368e+00,
    -1.292e-04, 2.392e+00, 7.855e-01, 0.05, 0.05,
]
FRANKA_ARM_EFFORT_LIMITS = [87, 87, 87, 87, 12, 12, 12, 100, 100]
FRANKA_ARM_ARMATURE = [0.195] * 4 + [0.074] * 3 + [0.1] * 2
FRANKA_ARM_KE = [4500, 4500, 3500, 3500, 2000, 2000, 2000, 100, 100]
FRANKA_ARM_KD = [450, 450, 350, 350, 200, 200, 200, 10, 10]


def build_franka_scene(include_table=True, include_cube=True, use_targets=True):
    """Build a Franka + table scene and optionally add a cube.

    Args:
        include_table: Whether to add a table.
        include_cube: Whether to add a cube on the table.
        use_targets: Whether to enable joint target gains for PD control.

    Returns:
        Tuple of (builder, cube_size, table_pos, cube_body).
    """
    builder = newton.ModelBuilder()
    builder.default_shape_cfg.gap = 0.0
    newton.solvers.SolverMuJoCo.register_custom_attributes(builder)

    table_height = 0.1
    table_pos = wp.vec3(0.0, -0.5, 0.5 * table_height)
    table_top_center = table_pos + wp.vec3(0.0, 0.0, 0.5 * table_height)
    if include_table:
        builder.add_shape_box(body=-1, hx=0.4, hy=0.4, hz=0.5 * table_height, xform=wp.transform(table_pos))

    # Load Franka from URDF (downloads from newton-assets if not cached)
    robot_base_pos = table_top_center + wp.vec3(-0.5, 0.0, 0.0)
    builder.add_urdf(
        str(newton.utils.download_asset("franka_emika_panda") / "urdf/fr3_franka_hand.urdf"),
        xform=wp.transform(robot_base_pos, wp.quat_identity()),
        floating=False,
        enable_self_collisions=False,
        parse_visuals_as_colliders=False,
    )

    # Set initial joint positions and PD gains
    builder.joint_q[:9] = FRANKA_HOME_Q
    if use_targets:
        builder.joint_target_pos[:9] = FRANKA_HOME_Q[:7] + [1.0, 1.0]
        builder.joint_target_ke[:9] = FRANKA_ARM_KE
        builder.joint_target_kd[:9] = FRANKA_ARM_KD
    else:
        builder.joint_target_pos[:9] = FRANKA_HOME_Q
        builder.joint_target_ke[:9] = [0] * 9
        builder.joint_target_kd[:9] = [0] * 9

    builder.joint_effort_limit[:9] = FRANKA_ARM_EFFORT_LIMITS
    builder.joint_armature[:9] = FRANKA_ARM_ARMATURE

    # Optional cube to pick
    cube_size = 0.05
    cube_body = None
    if include_cube:
        cube_pos = table_top_center + wp.vec3(0.0, 0.15, 0.5 * cube_size)
        cube_body = builder.add_body(xform=wp.transform(cube_pos, wp.quat_identity()))
        shape_cfg = newton.ModelBuilder.ShapeConfig(margin=1e-3, density=400.0)
        builder.add_shape_box(body=cube_body, hx=0.5 * cube_size, hy=0.5 * cube_size, hz=0.5 * cube_size, cfg=shape_cfg)

    return builder, cube_size, table_pos, cube_body


# Build a scene without cube or PD targets (we'll apply raw joint forces)
builder, _, _, _ = build_franka_scene(include_cube=False, use_targets=False)
model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

# Forward kinematics: compute body transforms from the initial joint_q
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

viewer = make_viewer("02_franka_scene")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)
viewer.begin_frame(0)
viewer.log_state(state_0)
viewer.end_frame()
viewer

Now let's apply some sinusoidal forces to the elbow joint:

In [ ]:
solver = newton.solvers.SolverMuJoCo(
    model,
    solver="newton",               # Constraint solver ("cg" or "newton")
    integrator="implicitfast",     # Integration method ("euler", "rk4", "implicit", "implicitfast")
    iterations=20,                 # Number of solver iterations
    ls_parallel=True,              # Enable parallel line search
    ls_iterations=100,             # Number of line-search iterations
    nconmax=1000,                  # Maximum number of contact points per world
    njmax=2000,                    # Maximum number of constraints per world
    cone="elliptic",               # Contact friction cone ("pyramidal" or "elliptic")
    impratio=1000.0,               # Frictional-to-normal constraint impedance ratio
    use_mujoco_contacts=False,     # Use Newton contacts instead of MuJoCo contacts
)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

viewer = make_viewer("03_franka_joint_forces")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

num_frames = 180
sim_time = 0.0
graph = None

print("Beginning simulation - it may take a minute or longer to compile the MuJoCo-Warp kernels...")


def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        model.collide(state_0, contacts)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


for frame in trange(num_frames):
    # Apply a sinusoidal torque to the elbow joint (index 2)
    joint_forces = np.zeros(model.joint_dof_count, dtype=np.float32)
    joint_forces[2] = 35.0 * np.sin(2.0 * np.pi * frame / num_frames)
    control.joint_f.assign(joint_forces)

    # Capture CUDA graph on first frame, then replay
    if graph is not None:
        wp.capture_launch(graph)
    elif wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph
    else:
        run_sim_substeps()

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.end_frame()
    sim_time += frame_dt

viewer

## 5) Joint Targets with Control.joint_target_pos

Most robot controllers drive joints with target positions (plus PD gains). Here we set a sinusoidal target on one joint using `Control.joint_target_pos`.

In [ ]:
builder, _, _, _ = build_franka_scene(include_cube=False, use_targets=True)
model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

# Same solver setup as before - see section 4 for parameter descriptions
solver = newton.solvers.SolverMuJoCo(
    model,
    solver="newton",
    integrator="implicitfast",
    iterations=20,
    ls_parallel=True,
    ls_iterations=100,
    nconmax=1000,
    njmax=2000,
    cone="elliptic",
    impratio=1000.0,
    use_mujoco_contacts=False,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

viewer = make_viewer("04_franka_joint_targets")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

base_target = model.joint_q.numpy().astype(np.float32)
base_target[7:9] = 0.04  # gripper finger targets

num_frames = 180
sim_time = 0.0
graph = None


def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        model.collide(state_0, contacts)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


for frame in trange(num_frames):
    # Oscillate joint 3 (shoulder/elbow) around its home position
    target = base_target.copy()
    target[3] = base_target[3] + 0.4 * np.sin(2.0 * np.pi * frame / num_frames)
    control.joint_target_pos.assign(target)

    if graph is not None:
        wp.capture_launch(graph)
    elif wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph
    else:
        run_sim_substeps()

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.end_frame()
    sim_time += frame_dt

viewer

## Inverse Kinematics (IK) in Newton

Newton's IK module `newton.ik` is a GPU-first, batched inverse-kinematics system built on Warp that lets you compose multiple objectives into a single solve, then choose the optimization strategy (`LM` or `LBFGS`), Jacobian backend (`ANALYTIC`, `AUTODIFF`, or `MIXED`), and optional multi-seed sampling (`NONE`, `GAUSS`, `UNIFORM`, `ROBERTS`). Because it solves many IK problems in parallel and can leverage CUDA graph execution for low-overhead repeated solves, Newton scales especially well in large batched workloads, delivering throughput and solve quality that are competitive with modern GPU IK libraries such as PyRoKi and cuRobo.

In [ ]:
ik_module_diagram = r"""
flowchart TD

    IKS[IKSolver]

    IKS --> OBJ[Objectives]
    OBJ --> O1[Position]
    OBJ --> O2[Rotation]
    OBJ --> O3[JointLimit]

    IKS --> OPT[Optimizer]
    OPT --> OPT1[Levenberg-Marquardt]
    OPT --> OPT2[LBFGS]

    IKS --> JAC[Jacobian mode]
    JAC --> J1[ANALYTIC]
    JAC --> J2[AUTODIFF]
    JAC --> J3[MIXED]
"""

render_mermaid(ik_module_diagram, theme="forest", line_color="#76b900")

## 6) IK Path Following

We build IK in very small visual steps so each new concept has a clear effect.
Joint limits stay enabled in every IK solve.

The code is intentionally repetitive in each step so every IK component is visible in place.

1. Solve a **single position target** (position + joint limits only).
2. Add a **rotation target** to the same single-point task.
3. Preview a rectangle target path (no IK yet).
4. Track the full rectangle with IK.

### Step 1: Position-Only IK on One Target

Start with the simplest IK task: one fixed end-effector position target.
Only the position objective + joint limits are active here.

In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

solver = newton.solvers.SolverMuJoCo(
    model, solver="newton", integrator="implicitfast", iterations=20,
    ls_parallel=True, ls_iterations=100, nconmax=500, njmax=1000,
    cone="elliptic", impratio=1000.0,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

ee_index = 11  # end-effector body index in the Franka model
home_pos_np = state_0.body_q.numpy()[ee_index][:3].astype(np.float32)
single_target_np = home_pos_np + np.array([0.0, 0.52, 0.04], dtype=np.float32)

# Position-only objective: drive the EE to a target point
pos_obj = ik.IKObjectivePosition(
    link_index=ee_index,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array([single_target_np], dtype=wp.vec3),
)

# Joint-limit objective keeps solutions within valid range
joint_limit_obj = ik.IKObjectiveJointLimit(
    joint_limit_lower=model.joint_limit_lower,
    joint_limit_upper=model.joint_limit_upper,
)

joint_q_ik = wp.clone(model.joint_q.reshape((1, -1)))
ik_solver = ik.IKSolver(
    model=model,
    n_problems=1,
    objectives=[pos_obj, joint_limit_obj],
    lambda_initial=0.1,
    jacobian_mode=ik.IKJacobianType.ANALYTIC,
)

viewer = make_viewer("05_franka_ik_single_position")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

sim_time = 0.0
ee_trace = []
graph = None


def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


for _ in trange(120):
    pos_obj.set_target_positions(wp.array([single_target_np], dtype=wp.vec3))
    ik_solver.step(joint_q_ik, joint_q_ik, iterations=24)

    # Send IK solution to the arm joints, keep gripper closed
    joint_target_pos_view = control.joint_target_pos.reshape((1, -1))
    wp.copy(dest=joint_target_pos_view[:, :7], src=joint_q_ik[:, :7])
    wp.copy(dest=joint_target_pos_view[:, 7:9], src=wp.array([[0.0, 0.0]], dtype=wp.float32))

    if graph is not None:
        wp.capture_launch(graph)
    elif wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph
    else:
        run_sim_substeps()

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)

    # Visualize the EE trace
    ee_trace.append(state_0.body_q.numpy()[ee_index][:3].astype(np.float32))
    if len(ee_trace) > 1:
        ee_trace_np = np.asarray(ee_trace, dtype=np.float32)
        viewer.log_lines(
            "/ik/single_pos_only/trace",
            ee_trace_np[:-1], ee_trace_np[1:],
            colors=(0.1, 0.8, 1.0), width=0.03,
        )

    viewer.end_frame()
    sim_time += frame_dt

viewer

### Step 2: Add Orientation on the Same Target

Keep the same single position target, but now also constrain end-effector orientation.
This isolates the effect of adding the rotation objective.

In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

solver = newton.solvers.SolverMuJoCo(
    model, solver="newton", integrator="implicitfast", iterations=20,
    ls_parallel=True, ls_iterations=100, nconmax=500, njmax=1000,
    cone="elliptic", impratio=1000.0,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

ee_index = 11
fixed_rot = wp.quat_from_axis_angle(wp.vec3(1.0, 0.0, 0.0), wp.pi / 2)
home_pos_np = state_0.body_q.numpy()[ee_index][:3].astype(np.float32)
single_target_np = home_pos_np + np.array([0.0, 0.22, 0.04], dtype=np.float32)

# Position + rotation objectives
pos_obj = ik.IKObjectivePosition(
    link_index=ee_index,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array([single_target_np], dtype=wp.vec3),
)
rot_obj = ik.IKObjectiveRotation(
    link_index=ee_index,
    link_offset_rotation=wp.quat_identity(),
    target_rotations=wp.array([fixed_rot[:4]], dtype=wp.vec4),
)

joint_limit_obj = ik.IKObjectiveJointLimit(
    joint_limit_lower=model.joint_limit_lower,
    joint_limit_upper=model.joint_limit_upper,
)

joint_q_ik = wp.clone(model.joint_q.reshape((1, -1)))
ik_solver = ik.IKSolver(
    model=model,
    n_problems=1,
    objectives=[pos_obj, rot_obj, joint_limit_obj],
    lambda_initial=0.1,
    jacobian_mode=ik.IKJacobianType.ANALYTIC,
)

viewer = make_viewer("06_franka_ik_single_pos_with_rot")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

sim_time = 0.0
ee_trace = []
graph = None


def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


for _ in trange(120):
    pos_obj.set_target_positions(wp.array([single_target_np], dtype=wp.vec3))
    rot_obj.set_target_rotations(wp.array([fixed_rot[:4]], dtype=wp.vec4))
    ik_solver.step(joint_q_ik, joint_q_ik, iterations=24)

    joint_target_pos_view = control.joint_target_pos.reshape((1, -1))
    wp.copy(dest=joint_target_pos_view[:, :7], src=joint_q_ik[:, :7])
    wp.copy(dest=joint_target_pos_view[:, 7:9], src=wp.array([[0.0, 0.0]], dtype=wp.float32))

    if graph is not None:
        wp.capture_launch(graph)
    elif wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph
    else:
        run_sim_substeps()

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)

    ee_trace.append(state_0.body_q.numpy()[ee_index][:3].astype(np.float32))
    if len(ee_trace) > 1:
        ee_trace_np = np.asarray(ee_trace, dtype=np.float32)
        viewer.log_lines(
            "/ik/single_with_rot/trace",
            ee_trace_np[:-1], ee_trace_np[1:],
            colors=(0.2, 1.0, 0.3), width=0.03,
        )

    viewer.end_frame()
    sim_time += frame_dt

viewer

### Step 3: Preview the Rectangle Path (No IK Yet)

Before solving IK on the rectangle, inspect the geometric target by itself.

In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()

state_0 = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

ee_index = 11
home_pos_np = state_0.body_q.numpy()[ee_index][:3].astype(np.float32)
rect_center = home_pos_np + np.array([0.0, 0.25, 0.0], dtype=np.float32)
rect_half = 0.08
rect_corners_np = np.array(
    [
        [rect_center[0] - rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] + rect_half, rect_center[2]],
        [rect_center[0] - rect_half, rect_center[1] + rect_half, rect_center[2]],
    ],
    dtype=np.float32,
)

viewer = make_viewer("07_franka_rectangle_preview")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

# Draw the rectangle target path as line segments
rect_starts_np = rect_corners_np
rect_ends_np = np.roll(rect_corners_np, -1, axis=0)

viewer.begin_frame(0.0)
viewer.log_state(state_0)
viewer.log_lines("/ik/rectangle_preview", rect_starts_np, rect_ends_np, colors=(1.0, 0.4, 0.0), width=0.012)
viewer.end_frame()

viewer

### Step 4: Full Rectangle IK Tracking

Now combine all objectives (position + orientation + joint limits) to follow the complete rectangle path.

In [ ]:
builder, _, _, _ = build_franka_scene(include_table=False, include_cube=False, use_targets=True)
model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

solver = newton.solvers.SolverMuJoCo(
    model, solver="newton", integrator="implicitfast", iterations=20,
    ls_parallel=True, ls_iterations=100, nconmax=500, njmax=1000,
    cone="elliptic", impratio=1000.0,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

ee_index = 11
fixed_rot = wp.quat_from_axis_angle(wp.vec3(1.0, 0.0, 0.0), wp.pi)
home_pos_np = state_0.body_q.numpy()[ee_index][:3].astype(np.float32)
rect_center = home_pos_np + np.array([0.0, 0.25, 0.0], dtype=np.float32)
rect_half = 0.08
edge_frames = 45  # frames per edge of the rectangle
rect_corners_np = np.array(
    [
        [rect_center[0] - rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] - rect_half, rect_center[2]],
        [rect_center[0] + rect_half, rect_center[1] + rect_half, rect_center[2]],
        [rect_center[0] - rect_half, rect_center[1] + rect_half, rect_center[2]],
    ],
    dtype=np.float32,
)

# Interpolate points along the rectangle edges
rect_path_points = []
for i in range(len(rect_corners_np)):
    start = rect_corners_np[i]
    end = rect_corners_np[(i + 1) % len(rect_corners_np)]
    for t in np.linspace(0.0, 1.0, edge_frames, endpoint=False):
        rect_path_points.append(start * (1.0 - t) + end * t)
rect_path_np = np.array(rect_path_points, dtype=np.float32)

# IK objectives: position + rotation + joint limits
pos_obj = ik.IKObjectivePosition(
    link_index=ee_index,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array([home_pos_np], dtype=wp.vec3),
)
rot_obj = ik.IKObjectiveRotation(
    link_index=ee_index,
    link_offset_rotation=wp.quat_identity(),
    target_rotations=wp.array([fixed_rot[:4]], dtype=wp.vec4),
)
joint_limit_obj = ik.IKObjectiveJointLimit(
    joint_limit_lower=model.joint_limit_lower,
    joint_limit_upper=model.joint_limit_upper,
)

joint_q_ik = wp.clone(model.joint_q.reshape((1, -1)))
ik_solver = ik.IKSolver(
    model=model,
    n_problems=1,
    objectives=[pos_obj, rot_obj, joint_limit_obj],
    lambda_initial=0.1,
    jacobian_mode=ik.IKJacobianType.ANALYTIC,
)

viewer = make_viewer("08_franka_ik_rectangle_full")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

# Precompute the target path line segments for visualization
target_starts_np = rect_path_np[:-1]
target_ends_np = rect_path_np[1:]

sim_time = 0.0
ee_trace = []
graph = None


def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


for p in tqdm(rect_path_np):
    target_wp = wp.vec3(p[0], p[1], p[2])
    pos_obj.set_target_positions(wp.array([target_wp], dtype=wp.vec3))
    rot_obj.set_target_rotations(wp.array([fixed_rot[:4]], dtype=wp.vec4))
    ik_solver.step(joint_q_ik, joint_q_ik, iterations=24)

    joint_target_pos_view = control.joint_target_pos.reshape((1, -1))
    wp.copy(dest=joint_target_pos_view[:, :7], src=joint_q_ik[:, :7])
    wp.copy(dest=joint_target_pos_view[:, 7:9], src=wp.array([[0.0, 0.0]], dtype=wp.float32))

    if graph is not None:
        wp.capture_launch(graph)
    elif wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph
    else:
        run_sim_substeps()

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_lines("/ik/rectangle_full/target", target_starts_np, target_ends_np, colors=(1.0, 0.4, 0.0), width=0.01)

    ee_trace.append(state_0.body_q.numpy()[ee_index][:3].astype(np.float32))
    if len(ee_trace) > 1:
        ee_trace_np = np.asarray(ee_trace, dtype=np.float32)
        viewer.log_lines(
            "/ik/rectangle_full/trace",
            ee_trace_np[:-1], ee_trace_np[1:],
            colors=(0.1, 0.8, 1.0), width=0.03,
        )

    viewer.end_frame()
    sim_time += frame_dt

viewer

## 7) Manipulation: Franka Pick-and-Place with IK + Gripper Control

We will build a simple manipulation scene inspired by `newton/examples/ik/example_ik_cube_stacking.py`, but simplified to **one cube** and **one world**. The flow:

1. Build a Franka robot + table + cube
2. Create a `SolverMuJoCo` for stable contact handling
3. Reuse the IK helpers from the previous section
4. Run a small pick-and-place sequence with gripper control

In [ ]:
builder, cube_size, table_pos, cube_body = build_franka_scene(include_cube=True, use_targets=True)
model = builder.finalize()

### SolverMuJoCo, State, Control, Contacts

The MuJoCo solver is robust for rigid-body contact and joints. We allocate a reusable `Contacts` buffer via `model.contacts()` and populate it each step with `model.collide()`.

In [ ]:
state_0 = model.state()
state_1 = model.state()
control = model.control()
contacts = model.contacts()

solver = newton.solvers.SolverMuJoCo(
    model,
    solver="newton",
    integrator="implicitfast",
    iterations=20,
    ls_parallel=True,
    ls_iterations=100,
    nconmax=1000,
    njmax=2000,
    cone="elliptic",
    impratio=1000.0,
    use_mujoco_contacts=False,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

fps = 60
frame_dt = 1.0 / fps
sim_substeps = 10
sim_dt = frame_dt / sim_substeps

### IK Setup (Reuse from Previous Section)

We reuse the IK helpers to drive the end-effector while we also control the gripper targets.

In [ ]:
def create_ik_system(model, state, ee_index, fixed_rot, use_orientation=True):
    """Create IK objectives + solver for one end-effector target."""
    home_pos_np = state.body_q.numpy()[ee_index][:3].astype(np.float32)
    home_pos = wp.vec3(home_pos_np[0], home_pos_np[1], home_pos_np[2])

    pos_obj = ik.IKObjectivePosition(
        link_index=ee_index,
        link_offset=wp.vec3(0.0, 0.0, 0.0),
        target_positions=wp.array([home_pos], dtype=wp.vec3),
    )

    objectives = [pos_obj]
    rot_obj = None
    if use_orientation:
        rot_obj = ik.IKObjectiveRotation(
            link_index=ee_index,
            link_offset_rotation=wp.quat_identity(),
            target_rotations=wp.array([fixed_rot[:4]], dtype=wp.vec4),
        )
        objectives.append(rot_obj)

    objectives.append(
        ik.IKObjectiveJointLimit(
            joint_limit_lower=model.joint_limit_lower,
            joint_limit_upper=model.joint_limit_upper,
        )
    )

    joint_q_ik = wp.clone(model.joint_q.reshape((1, -1)))
    ik_solver = ik.IKSolver(
        model=model,
        n_problems=1,
        objectives=objectives,
        lambda_initial=0.1,
        jacobian_mode=ik.IKJacobianType.ANALYTIC,
    )

    return pos_obj, rot_obj, joint_q_ik, ik_solver, 24, home_pos_np


def apply_ik_target(pos_obj, rot_obj, ik_solver, joint_q_ik, control, target_pos, fixed_rot=None, ik_iters=24, gripper=None):
    """Run one IK step and write the solution into the control targets."""
    pos_obj.set_target_positions(wp.array([target_pos], dtype=wp.vec3))

    if rot_obj is not None:
        if fixed_rot is None:
            raise ValueError("fixed_rot must be provided when rot_obj is used.")
        rot_obj.set_target_rotations(wp.array([fixed_rot[:4]], dtype=wp.vec4))

    ik_solver.step(joint_q_ik, joint_q_ik, iterations=ik_iters)

    joint_target_pos_view = control.joint_target_pos.reshape((1, -1))
    wp.copy(dest=joint_target_pos_view[:, :7], src=joint_q_ik[:, :7])

    grip = 0.0 if gripper is None else float(gripper)
    wp.copy(dest=joint_target_pos_view[:, 7:9], src=wp.array([[grip, grip]], dtype=wp.float32))

### Pick-and-Place Sequence

We will define a short sequence of end-effector targets:

1. Approach above the cube
2. Move down to grasp
3. Close gripper
4. Lift
5. Move to drop-off
6. Open gripper
7. Retract to home

In [ ]:
ee_index = 11
fixed_rot = wp.quat_from_axis_angle(wp.vec3(1.0, 0.0, 0.0), wp.pi)

pos_obj, rot_obj, joint_q_ik, ik_solver, ik_iters, home_pos_np = create_ik_system(
    model=model,
    state=state_0,
    ee_index=ee_index,
    fixed_rot=fixed_rot,
)

In [ ]:
viewer = make_viewer("final_franka_pick_place")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

# Key positions for the pick-and-place sequence
cube_pos_np = state_0.body_q.numpy()[cube_body][:3]
cube_pos = wp.vec3(cube_pos_np[0], cube_pos_np[1], cube_pos_np[2])

approach_offset = wp.vec3(0.0, 0.0, 3.0 * cube_size)
lift_offset = wp.vec3(0.0, 0.0, 4.0 * cube_size)
table_height = 0.1
table_top_center = table_pos + wp.vec3(0.0, 0.0, 0.5 * table_height)
drop_pos = table_top_center + wp.vec3(0.0, -0.15, 0.5 * cube_size)

graph = None


def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        model.collide(state_0, contacts)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0


def step_sim():
    """Run substeps, capturing a CUDA graph on first call for efficiency."""
    global graph
    if graph is not None:
        wp.capture_launch(graph)
    elif wp.get_device().is_cuda:
        with wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph
    else:
        run_sim_substeps()


def run_phase(target_pos, grip, frames, sim_time):
    """Execute one phase of the pick-and-place, optionally interpolating the gripper."""
    if isinstance(grip, tuple):
        grip_start, grip_end = float(grip[0]), float(grip[1])
    else:
        grip_start = grip_end = float(grip)

    for i in range(frames):
        alpha = (i + 1) / frames if frames > 1 else 1.0
        grip_i = grip_start + (grip_end - grip_start) * alpha
        apply_ik_target(
            pos_obj=pos_obj, rot_obj=rot_obj,
            ik_solver=ik_solver, joint_q_ik=joint_q_ik,
            control=control, target_pos=target_pos,
            fixed_rot=fixed_rot, ik_iters=ik_iters, gripper=grip_i,
        )
        step_sim()

        viewer.begin_frame(sim_time)
        viewer.log_state(state_0)
        viewer.end_frame()
        sim_time += frame_dt

    return sim_time


def record_sequence():
    sim_time = 0.0
    phase_bar = tqdm(sequence, desc="phase: init")
    for phase in phase_bar:
        phase_bar.set_description(f"phase: {phase['name']}")
        sim_time = run_phase(phase["pos"], phase["grip"], int(phase["frames"]), sim_time)


grip_open = 0.06
grip_closed = 0.0

# Each phase: move to a target position with a gripper state, over N frames
sequence = [
    {"name": "approach", "pos": cube_pos + approach_offset, "grip": grip_open, "frames": 60},
    {"name": "descend",  "pos": cube_pos,                   "grip": grip_open, "frames": 60},
    {"name": "close",    "pos": cube_pos,                   "grip": (grip_open, grip_closed), "frames": 60},
    {"name": "lift",     "pos": cube_pos + lift_offset,     "grip": grip_closed, "frames": 60},
    {"name": "move",     "pos": drop_pos + approach_offset, "grip": grip_closed, "frames": 60},
    {"name": "drop",     "pos": drop_pos,                   "grip": (grip_closed, grip_open), "frames": 60},
    {"name": "retract",  "pos": drop_pos + approach_offset, "grip": grip_open, "frames": 60},
]

record_sequence()
viewer

## Wrap-Up and Next Steps

You now have a full path from high-level concepts to a working manipulation scene. To explore more:

- Newton GitHub repository: https://github.com/newton-physics/newton
- Overview: https://newton-physics.github.io/newton/guide/overview.html